# H-010 · Does Short-Selling Pressure Predict Forward Returns?

Abnormal FINRA off-exchange short volume (own-history z-score) as a cross-sectional signal. Supporting SEC filing-clock columns condition the GBM on event proximity.

**Expected standalone IC ~0.020–0.035 at 5d** — kept for orthogonality, not a 0.05 hero factor.

Evaluation uses the S1 **trade-date** panel: Alphalens pivots `open` (no `shift(-1)`); labels are open-to-open. Prior close-to-close ICs are not comparable.

**FINRA fetch:** cold pulls show a month-level `tqdm` bar on stderr (`FINRA short vol`). Year caches and empty-day sidecars live under `01_data/cache/finra_short_volume/` (`{FACILITY}_{YYYY}.parquet`, `{FACILITY}_{YYYY}_empty.parquet`).

## Short volume vs short interest

Short *volume* is gross daily shorting activity from off-exchange trade reports (FINRA). Short *interest* is a bi-monthly position snapshot. They are related but not directly reconcilable (FINRA Information Notice 05/10/19). Raw short/total is inflated by wholesaler internalisation of retail buys; own-history z-scoring strips that structural level.

## 10-Q / 10-K vs 8-K

The earnings press release is typically an **8-K Item 2.02** (where the stock gaps / PEAD is measured). The **10-Q/10-K** is the full XBRL filing days–weeks later. CompanyFacts `filed` dates are 10-Q/10-K — correct for fundamental staleness, a lagged proxy for earnings proximity. 8-K Item 2.02 sourcing is deferred.


## 0. Imports & Config


In [1]:
import os
import sys

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

import numpy as np
import pandas as pd
import alphalens as al

from data.processing.feature_store import add_short_flow_factors

ROOT = os.path.abspath(os.getcwd())
while not os.path.isdir(os.path.join(ROOT, "01_data", "ingestion")):
    parent = os.path.dirname(ROOT)
    if parent == ROOT:
        break
    ROOT = parent
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

TRAIN_PANEL_PATH = os.path.join(
    ROOT, "01_data", "data_files", "s1_equities", "s1_factor_panel_train.parquet"
)
TEARSHEET_DIR = os.path.join(
    ROOT, "02_research", "notebooks", "factor_tests", "tearsheets"
)
SHORT_FLOW_CACHE_PATH = os.path.join(
    ROOT, "01_data", "data_files", "s1_equities", "s1_h010_short_flow.parquet"
)
FILING_CLOCK_CACHE_PATH = os.path.join(
    ROOT, "01_data", "data_files", "s1_equities", "s1_h010_filing_clock.parquet"
)

FORCE_REBUILD = False
SMOOTH_WINDOWS = [1, 5, 10]
BASELINE_WINDOWS = [60, 120]
PERIODS = (1, 5, 21)
QUANTILES = 5
MAX_LOSS = 0.35


## 1. Data Loading


In [2]:
panel = pd.read_parquet(TRAIN_PANEL_PATH)
required = {"date", "ticker", "open", "close", "feature_date"}
missing = required - set(panel.columns)
if missing:
    raise ValueError(f"train panel missing columns: {sorted(missing)}")
if not panel["feature_date"].lt(panel["date"]).all():
    raise ValueError("feature_date must be strictly before date on every row")

panel = panel.copy()
panel["date"] = pd.to_datetime(panel["date"])
panel["feature_date"] = pd.to_datetime(panel["feature_date"])
panel["ticker"] = panel["ticker"].astype(str).str.strip().str.upper()
tickers = sorted(panel["ticker"].unique().tolist())
start = panel["feature_date"].min().date()
end = panel["feature_date"].max().date()
print(
    f"train panel: rows={len(panel):,}  tickers={len(tickers)}  "
    f"trade dates={panel['date'].nunique():,}  feature [{start} -> {end}]"
)
print(
    "FINRA short volume and SEC filing-clock anchors are attached inside "
    "add_short_flow_factors (short_volume_data_exists / filing_clock_data_exists)."
)
panel.head()


train panel: rows=289,381  tickers=100  trade dates=2,915  feature [2010-01-04 -> 2021-08-02]
SHORT FLOW CACHE MISS: fetching FINRA [2010-01-04 -> 2021-08-02]


FINRA HTTP error FNSQ 20100118: 403 Client Error: Forbidden for url: https://cdn.finra.org/equity/regsho/daily/FNSQshvol20100118.txt
FINRA HTTP error FNSQ 20100215: 403 Client Error: Forbidden for url: https://cdn.finra.org/equity/regsho/daily/FNSQshvol20100215.txt
FINRA HTTP error FNSQ 20100402: 403 Client Error: Forbidden for url: https://cdn.finra.org/equity/regsho/daily/FNSQshvol20100402.txt
FINRA HTTP error FNSQ 20100531: 403 Client Error: Forbidden for url: https://cdn.finra.org/equity/regsho/daily/FNSQshvol20100531.txt
FINRA HTTP error FNSQ 20100705: 403 Client Error: Forbidden for url: https://cdn.finra.org/equity/regsho/daily/FNSQshvol20100705.txt
FINRA HTTP error FNSQ 20100906: 403 Client Error: Forbidden for url: https://cdn.finra.org/equity/regsho/daily/FNSQshvol20100906.txt
FINRA HTTP error FNSQ 20101125: 403 Client Error: Forbidden for url: https://cdn.finra.org/equity/regsho/daily/FNSQshvol20101125.txt
FINRA HTTP error FNSQ 20101224: 403 Client Error: Forbidden for url: 

Wrote c:\Users\User\Desktop\ML Algorithmic Trading\Portfolio 26\trading_portfolio\01_data\data_files\s1_equities\s1_h010_short_flow.parquet rows=285,571
short_volume coverage=98.7%  unmatched tickers sample=[]


No SEC CIK for ticker AET; leaving filing-clock anchors NaT
No SEC CIK for ticker ESRX; leaving filing-clock anchors NaT
No SEC CIK for ticker TWX; leaving filing-clock anchors NaT


last_filed coverage=93.1%


,date,ticker,open,high,low,close,volume,feature_date,fwd_ret_1,fwd_ret_5,fwd_ret_21,short_volume,short_exempt_volume,total_volume,last_filed,expected_next_filed
0,2010-01-05,AAPL,6.424143,6.421146,6.357683,6.406478,493729600.0,2010-01-04,-0.001025,-0.025210,-0.083271,3266483.0,0.0,7428397.0,2009-10-27,2010-02-01
1,2010-01-06,AAPL,6.417558,6.453779,6.383729,6.417557,601904800.0,2010-01-05,-0.012268,-0.030367,-0.101456,3159681.0,0.0,7650995.0,2009-10-27,2010-02-01
2,2010-01-07,AAPL,6.338825,6.443003,6.308892,6.315478,552160000.0,2010-01-06,-0.006848,-0.007745,-0.075844,2784317.0,0.0,8182293.0,2009-10-27,2010-02-01
3,2010-01-08,AAPL,6.295419,6.346309,6.258000,6.303801,477131200.0,2010-01-07,0.011888,0.002996,-0.066001,2535802.0,0.0,7534274.0,2009-10-27,2010-02-01
4,2010-01-11,AAPL,6.370258,6.346310,6.258300,6.345711,447610800.0,2010-01-08,-0.016965,-0.021005,-0.079464,3183456.0,0.0,6860212.0,2009-10-27,2010-02-01


## 2. Data Cleaning & Engineering

Symbology coverage, 2018 facility-break stability (ratio distribution), event-clock coverage.


In [3]:
svr = panel["short_volume"] / panel["total_volume"]
panel["_svr_raw"] = svr.where(panel["total_volume"] > 0)
print("SVR describe:")
print(panel["_svr_raw"].describe())

cut = pd.Timestamp("2018-09-01")
pre = panel.loc[panel["feature_date"] < cut, "_svr_raw"]
post = panel.loc[panel["feature_date"] >= cut, "_svr_raw"]
print(f"pre-2018-09 SVR mean={pre.mean():.4f}  post mean={post.mean():.4f}")

if panel["last_filed"].notna().any():
    tmp = (panel["date"] - pd.to_datetime(panel["last_filed"])).dt.days
    print(f"days_since_filing describe:\n{tmp.describe()}")


SVR describe:
count    285569.000000
mean          0.415720
std           0.118899
min           0.024733
25%           0.331873
50%           0.411860
75%           0.496855
max           0.965665
Name: _svr_raw, dtype: float64
pre-2018-09 SVR mean=0.4169  post mean=0.4123
days_since_filing describe:
count    269315.000000
mean         47.949695
std          29.360403
min           1.000000
25%          23.000000
50%          47.000000
75%          70.000000
max         364.000000
dtype: float64


## 3. Modeling / Signal Construction


In [4]:
panel = add_short_flow_factors(
    panel,
    feature_subset=["abnormal"],
    smooth_window=SMOOTH_WINDOWS,
    baseline_window=BASELINE_WINDOWS,
    short_volume_data_exists=False)
panel = add_short_flow_factors(panel, feature_subset=["ratio"], short_volume_data_exists=True)
panel = add_short_flow_factors(panel, feature_subset=["filing_since"], filing_clock_data_exists=False)
panel = add_short_flow_factors(panel, feature_subset=["filing_expected_until"], filing_clock_data_exists=True)

FACTOR_COLS = [
    c
    for c in panel.columns
    if c.startswith("short_flow_") or c.startswith("filing_clock_")
]
print("factor columns:", FACTOR_COLS)
panel[FACTOR_COLS].describe()


factor columns: ['short_flow_abnormal_1_60', 'short_flow_abnormal_1_120', 'short_flow_abnormal_5_60', 'short_flow_abnormal_5_120', 'short_flow_abnormal_10_60', 'short_flow_abnormal_10_120', 'short_flow_ratio', 'filing_clock_since', 'filing_clock_expected_until']


,short_flow_abnormal_1_60,short_flow_abnormal_1_120,short_flow_abnormal_5_60,short_flow_abnormal_5_120,short_flow_abnormal_10_60,short_flow_abnormal_10_120,short_flow_ratio,filing_clock_since,filing_clock_expected_until
count,279449.000000,273329.000000,279041.000000,272921.000000,278531.000000,272411.000000,285569.000000,269315.000000,268328.000000
mean,0.005533,0.004574,0.004058,0.003324,0.003590,0.001859,0.415720,47.949695,46.963042
std,1.092406,1.069670,1.190718,1.138734,1.274689,1.193791,0.118899,29.360403,32.437610
min,-7.527491,-5.163679,-7.143284,-5.462408,-5.471514,-5.658567,0.024733,1.000000,-286.000000
25%,-0.752012,-0.745206,-0.818109,-0.785313,-0.891327,-0.829348,0.331873,23.000000,22.000000
50%,-0.014183,-0.016260,0.002538,-0.002373,0.003528,0.000978,0.411860,47.000000,45.000000
75%,0.747912,0.739059,0.825518,0.790548,0.899821,0.832050,0.496855,70.000000,70.000000
max,9.009393,6.271651,6.420366,6.878841,5.903167,5.805147,0.965665,364.000000,363.000000


## 4. Evaluation


### 4.1 IC / spread screen


In [5]:
def to_alphalens_prices(panel: pd.DataFrame) -> pd.DataFrame:
    """Wide open matrix for Alphalens (trade-date panel; entry at open)."""
    prices = panel.pivot(index="date", columns="ticker", values="open")
    prices.index = pd.to_datetime(prices.index)
    return prices.sort_index()


def to_alphalens_factor(panel: pd.DataFrame, col: str) -> pd.Series:
    s = panel.set_index(["date", "ticker"])[col].copy()
    s = s.sort_index()
    return s


def factor_screen_metrics(
    factor,
    prices,
    periods=PERIODS,
    quantiles=QUANTILES,
    max_loss=MAX_LOSS,
):
    factor_data = al.utils.get_clean_factor_and_forward_returns(
        factor=factor,
        prices=prices,
        quantiles=quantiles,
        periods=periods,
        max_loss=max_loss,
    )
    ic = al.performance.factor_information_coefficient(factor_data)
    out = {}
    for p in periods:
        matches = [c for c in ic.columns if str(p) in str(c)]
        out[f"ic_{p}d"] = float(ic[matches[0]].mean()) if matches else float("nan")
    return out


def run_full_tear(panel, factor_col, prices, tearsheet_dir=TEARSHEET_DIR):
    os.makedirs(tearsheet_dir, exist_ok=True)
    out_path = os.path.join(tearsheet_dir, f"H-010_{factor_col}.pdf")
    factor_data = al.utils.get_clean_factor_and_forward_returns(
        factor=to_alphalens_factor(panel, factor_col),
        prices=prices,
        quantiles=QUANTILES,
        periods=PERIODS,
        max_loss=MAX_LOSS,
    )
    pdf = PdfPages(out_path)
    _show = plt.show

    def _capture(*args, **kwargs):
        for num in plt.get_fignums():
            fig = plt.figure(num)
            if fig.axes:
                pdf.savefig(fig)
        plt.close("all")

    plt.show = _capture
    try:
        al.tears.create_full_tear_sheet(factor_data, long_short=True)
    finally:
        plt.show = _show
        pdf.close()
    print(f"tearsheet: {out_path}")
    return factor_data


prices = to_alphalens_prices(panel)
rows = []
for col in FACTOR_COLS:
    try:
        metrics = factor_screen_metrics(to_alphalens_factor(panel, col), prices)
        rows.append({"factor": col, **metrics})
    except Exception as e:
        rows.append({"factor": col, "error": str(e)})
summary = pd.DataFrame(rows)
if "ic_5d" in summary.columns:
    summary = summary.sort_values("ic_5d", ascending=True).reset_index(drop=True)
summary


c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\alphalens\utils.py:308: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  returns = prices.pct_change(period)
c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\alphalens\utils.py:308: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  returns = prices.pct_change(period)
c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\alphalens\utils.py:308: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any n

,factor,ic_1d,ic_5d,ic_21d
0,filing_clock_expected_until,-0.008574,-0.013439,-0.017219
1,short_flow_ratio,-0.013842,-0.010474,-0.006354
2,short_flow_abnormal_1_120,-0.012888,-0.007909,-0.002786
3,short_flow_abnormal_1_60,-0.013820,-0.007436,-0.002128
4,short_flow_abnormal_10_120,-0.002182,-0.001476,-0.004178
5,short_flow_abnormal_5_120,-0.003348,-0.001159,-0.002520
6,short_flow_abnormal_10_60,-0.002156,-0.000233,-0.002186
7,short_flow_abnormal_5_60,-0.003522,-0.000233,-0.000442
8,filing_clock_since,0.005332,0.007717,0.007686


### 4.2 Full tear sheet (manual pick)


In [6]:
# Edit after reviewing §4.1 (defaults are placeholders)
TEAR_FACTORS = [
    "filing_clock_expected_until",
    "short_flow_ratio",
    "short_flow_abnormal_1_120",
    "filing_clock_since",
]

for tear_col in TEAR_FACTORS:
    if tear_col not in panel.columns:
        cands = [c for c in FACTOR_COLS if c.startswith("short_flow_abnormal")]
        tear_col = cands[0] if cands else FACTOR_COLS[0]

    print(f"\n===== Tear sheet: {tear_col} =====")
    tear_data = run_full_tear(panel, tear_col, prices)
    


===== Tear sheet: filing_clock_expected_until =====


c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\alphalens\utils.py:308: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  returns = prices.pct_change(period)
c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\alphalens\utils.py:308: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  returns = prices.pct_change(period)
c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\alphalens\utils.py:308: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any n

Quantiles Statistics


,min,max,mean,std,count,count %
factor_quantile,,,,,,
1.0,-286.0,72.0,16.593582,21.889142,59584,22.402695
2.0,-20.0,99.0,39.110256,23.030439,55380,20.822054
3.0,-16.0,107.0,48.323921,24.199786,54930,20.652860
4.0,-8.0,112.0,57.458726,23.657525,47839,17.986750
5.0,27.0,363.0,82.231658,28.460711,48235,18.135640


Returns Analysis


...

c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\alphalens\performance.py:118: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  grouper.append(pd.Grouper(freq=by_time))
c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\alphalens\utils.py:928: UserWarning: Skipping return periods that aren't exact multiples of days.
  warnings.warn(


Turnover Analysis


,1D,5D,21D
Quantile 1.0 Mean Turnover,0.070,0.300,0.737
Quantile 2.0 Mean Turnover,0.077,0.321,0.761
Quantile 3.0 Mean Turnover,0.074,0.306,0.742
Quantile 4.0 Mean Turnover,0.073,0.306,0.749
Quantile 5.0 Mean Turnover,0.066,0.263,0.729


,1D,5D,21D
Mean Factor Rank Autocorrelation,0.924,0.666,0.105


tearsheet: c:\Users\User\Desktop\ML Algorithmic Trading\Portfolio 26\trading_portfolio\02_research\notebooks\factor_tests\tearsheets\H-010_filing_clock_since.pdf


## 5. Wrap-up / next steps

- Record variants tried vs best (pre-registration).
- Keep/kill rests on correlation to existing kept factors and nested GBM gain in H-011, not standalone IC.
- Re-run after FINRA cache is warm if first pass was coverage-limited.
